**Pattern LUT Thesis Plots**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import glob
from matplotlib.colors import LogNorm
import numpy.ma as ma
import matplotlib.patches as patches
import uproot
import awkward as ak
import matplotlib.cm as cm 
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import FuncFormatter, LogLocator
from collections import defaultdict

These are the scripts to reproduce all figures included in my thesis. Assuming you have (for a given LUT threshold) the following files: 

* a "clusters.txt"
* a "LUT.txt"
* a "XXX.ROOT" (contaning one/multiple branches for tracking)

everything should work. It is important that these files are located within the same directory!!

I worked with roughly 1mil events, and this was done with 100 input files of 10.000 events each. Each of these 100 files then underwent the process described in the README.md to produce the clusters.txt, LUT.txt, and XXX.ROOT files. This then left me with 100 clusters.txt files, 100 LUT.txt files, and 100 XXX.ROOT files. I then concatenated all these clusters.txt files together, which left me with one final "allclusters.txt" file. I didn't do this for the XXX.ROOT files but I'm sure you could and it would be simpler to make these plots. Anyway, the code here regards one big, final "allclusters.txt" file and 100 XXX.ROOT files. 



To reproduce Figs. 12a and 12b:

In [ ]:
#First, load your files.

truth_file = "new_cosmos_clusters.txt" #clusters.txt file as produced by ClusterTripletMaker
notruth_file = "cosmos_notruthclusters.txt" #clusters.txt file as produced by ClusterTripletMaker

def number_clusters(file):
    lines_in_file = open(file, 'r').readlines()
    number_of_lines = len(lines_in_file)
    return(number_of_lines)

print("Number of cluster triplets (truth): ", number_clusters(truth_file))
print("Number of cluster triplets (nontruth): ", number_clusters(notruth_file))

In [ ]:
#Select if you would like to reproduce 12a (truth case) or 12b (non-truth/all/unfiltered case)

truth_filtering = False #to reproduce 12a (truth_filtering = True) or 12b (truth_filtering = False)

In [ ]:
#This will print a summary of your "pattern coordinates" as well as the plot(s)
#First, load your files.

dfs = []

if truth_filtering == True: 
    clusters_file = truth_file
    number_of_clusters = number_clusters(truth_file)
    title = "Track Candidates (Truth Clusters Only)"
if truth_filtering == False: 
    clusters_file = notruth_file
    number_of_clusters = number_clusters(notruth_file)
    title = "Track Candidates (All Clusters)"

df = pd.read_csv(clusters_file, sep="\s+", names=["ev", "pad 1", "pad 2", "pad 3"], engine='python')
x = df["pad 2"] - df["pad 1"]
y = df["pad 3"] - df["pad 2"]
dfs.append(pd.DataFrame({"x": x, "y": y}))

all_data = pd.concat(dfs, ignore_index=True)
counts = all_data.groupby(["x", "y"]).size()
print("Pattern coordinate distribution: \n", counts)
print()

plt.rcParams.update({'font.size': 8})
x_bins = np.array([ #-10.25,-9.75,-9.25,-8.75,-8.25,-7.75, #you can adjust these bins to "zoom in" or "zoom out"
                   #-7.25,-6.75,-6.25,-5.75,
                   -5.25,-4.75,-4.25,
                   -3.75,-3.25,-2.75,-2.25,-1.75, -1.25, -0.75,
                   -0.25, 0.25, 0.75, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75,
                   4.25,  4.75,5.25,
                   #5.75, 6.25,6.75,7.25,7.75,
                   #8.25,8.75,9.25,9.75,10.25
                   ])
y_bins = np.array([ #-10.25,-9.75,-9.25,-8.75,-8.25,-7.75,
                   #-7.25,-6.75,-6.25,-5.75,
                   -5.25,-4.75,-4.25,
                   -3.75,-3.25,-2.75,-2.25,-1.75, -1.25, -0.75,
                   -0.25, 0.25, 0.75, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75,
                   4.25,  4.75,5.25,
                   #5.75, 6.25,6.75,7.25,7.75,
                   #8.25,8.75,9.25,9.75,10.25
                   ])
H, xedges, yedges = np.histogram2d(all_data["x"], all_data["y"], bins=[x_bins, y_bins])
fig, ax = plt.subplots()
H[H == 0] = 1e-1  # small value so log works
im = ax.imshow(H.T, origin="lower", norm=LogNorm(vmin=1, vmax=H.max()))
cbar = plt.colorbar(im, ax=ax)
cbar.set_label = ("label")
text = False

x_centers = [ #-10,-9.5,-9,-8.5,-8,-7.5, #if you adjust the bins make sure to adjust the centers too
             #-7,-6.5,-6,-5.5,
             -5,-4.5,-4,
             -3.5,-3,-2.5,-2,-1.5,-1,-0.5, 0,
             0.5,1,1.5,2,2.5,3,3.5,
             4,4.5,5,
             #5.5, 6, 6.5, 7,
             #7.5, 8, 8.5, 9, 9.5, 10
             ]
y_centers = [ #-10,-9.5,-9,-8.5,-8,-7.5,
             #-7,-6.5,-6,-5.5,
             -5,-4.5,-4,
             -3.5,-3,-2.5,-2,-1.5,-1,-0.5,0,
             0.5, 1, 1.5, 2, 2.5, 3, 3.5,
             4, 4.5,5,
             #5.5, 6, 6.5, 7,
             #7.5,8,8.5,9,9.5,10
             ]

ax.set_xticks(range(len(x_centers)), labels=x_centers, rotation = 45, rotation_mode= "anchor")
ax.set_yticks(range(len(y_centers)), labels=y_centers)

for i in range(len(x_centers)):
    for j in range(len(y_centers)):
        val = H.T[j,i]
        if text == True:
           if val > 0:
               if val <= H.T.max() * 0.1:
                   ax.text(i, j, int(H.T[j, i]),
                           ha="center", va="center", color="w")
                   if val > H.T.max() * 0.1:
                       ax.text(i, j, int(H.T[j, i]),
                               ha="center", va="center", color="black") 

ax.set_xlabel(r"Pad 1$\rightarrow$2 $(p_2-p_1)$")
ax.set_ylabel(r"Pad 2$\rightarrow$3 $(p_3-p_2)$")
ax.set_title(title)

ax.text(0.98,0.98, f"n = {number_of_clusters}", transform = ax.transAxes,
        verticalalignment = "top",horizontalalignment = "right", bbox = dict(boxstyle = "round", facecolor= "white", edgecolor = "black"))

Then to reproduce Figs. 13 and 14:

In [ ]:
#This block will load both files and then print the information needed to produce the plot(s)

def process_groups(input_file):
    with open(input_file, "r") as infile:
        groups = defaultdict(list)
        total_lines = 0
        for line in infile:
            if not line.strip():
                continue
            ev, p1, p2, p3 = map(float, line.split())
            p12 = p2 - p1
            p23 = p3 - p2
            groups[(p12, p23)].append((ev, p1, p2, p3))
            total_lines += 1
    sorted_groups = sorted(groups.items(), key=lambda g: len(g[1]), reverse=True)
    percentages = []

    for (p12, p23), tracks in sorted_groups:
        count = len(tracks)
        frac = count / total_lines * 100
        percentages.append(frac)
    return (total_lines, groups, percentages)

truth = process_groups(truth_file)
notruth = process_groups(notruth_file)

print("------------ TRUTH -----------")
print("Total number of clusters (truth case): ", truth[0])
print("Total number of patterns: ", len(truth[1]))
#print("Full list of each pattern's frequency: ", truth[2]) #you can print this if you want to look at it, but depending on your number 
                                                            #of events, it might be a veeery long list    

print("\n-----------NOTRUTH ---------")
print("Total number of clusters (nontruth case): ", notruth[0])
print("Total number of patterns: ", len(notruth[1]))
#print("Full list of each pattern's frequency: ", notruth[2]) #same here as 6 lines up 

In [ ]:
#Just like for 12a/12b select if you want to use the truth-filtering to produce either Fig. 13 or 14

truth_level = False #True for Fig. 13 and False for Fig. 14 

In [ ]:
#produces the plot!

if truth_level == True:
    tot = truth[0]
    freqs = truth[2]
    title = "Frequencies of Pattern Groups (Truth Clusters)"
if truth_level == False:
    tot = notruth[0]
    freqs = notruth[2]
    title = "Frequencies of Pattern Groups (All Clusters)"

percentages = np.array(freqs)
sorted_percentages = np.sort(percentages)[::-1]
cumulative = np.cumsum(sorted_percentages)

fig, ax1 = plt.subplots(figsize=(12,6))

norm = mcolors.LogNorm(vmin=min(sorted_percentages), vmax=max(sorted_percentages))
cmap = cm.viridis
colors = cmap(norm(sorted_percentages))

#pattern group frequencies
ax1.bar(range(len(sorted_percentages)), sorted_percentages,
    	color=colors)
ax1.set_yscale("log")
ax1.set_ylabel("Fraction per group (%)")
ax1.set_xlabel(f"Number of groups (Total {len(freqs)})")
ax1.yaxis.set_major_locator(LogLocator(base=10))
ax1.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = plt.colorbar(sm, ax=ax1, location="left", pad=0.12)
cbar.ax.yaxis.set_major_locator(LogLocator(base=10))
cbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))

#cumulative frac
ax2 = ax1.twinx()
ax2.plot(cumulative, color="red", marker="o")
ax2.set_yscale("log")
ax2.set_ylabel("Cumulative fraction (%)", color="red")
ax2.tick_params(axis='y', colors='red')

ax1.text(0.98, 0.91, f"$n = {tot}$",
    transform=ax1.transAxes,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"))

plt.title(title) 
plt.grid()
plt.show()

To generate Figs. 15 and 16: 

In [ ]:
#this block loads your files and prints the data relevant to the plot(s)

#AGAIN: directory locations are important here. If you are using files which are located on an SSH (lunarc) then you will need to 
#       move this notebook to that directory in order for it to be able to read those files. If you run it locally this cell will 
#       just print a bunch of zeros since it won't have any files to read.  

notruth_base_path = "/projects/hep/fs9/shared/ldmx/users/hansalin/lucia/notruthfig/tracks"
truth_base_path = "/projects/hep/fs9/shared/ldmx/users/hansalin/lucia/tracks"

truth_threshold_files = {
    "0p0":    glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p0.root"),
    "0p0002": glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p0002.root"),
    "0p0004": glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p0004.root"),
    "0p0005": glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p0005.root"),
    "0p0008": glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p0008.root"),
    "0p0015": glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p0015.root"),
    "0p005":  glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p005.root"),
    "0p06":   glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p06.root"),
    "0p5":    glob.glob(f"{truth_base_path}/track_lut_truthisolater_*_clusters_thr_0p5.root")
} #change to whatever your files are called

notruth_threshold_files = {
    "0p0":    glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p0.root"),
    "0p0002": glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p0002.root"),
    "0p0004": glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p0004.root"),
    "0p0005": glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p0005.root"),
    "0p0008": glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p0008.root"),
    "0p0015": glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p0015.root"),
    "0p005":  glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p005.root"),
    "0p06":   glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p06.root"),
    "0p5":    glob.glob(f"{notruth_base_path}/track_lut_nontruthclusters_*_thr_0p5.root"),
}

tree_name = "LDMX_Events"

def process_thresholds(threshold_files):
    clusters_length = []
    lut_tracks_made = []
    lut_fakes_2 = []
    lut_fakes_3 = []
    total_events = 0
    tstp_fakes = 0
    tstp_tracks_made = 0

    for threshold, file_list in threshold_files.items():
        tstp_tracks = []
        lut_tracks = []

        for filename in file_list:
            with uproot.open(filename) as f:
                tree = f[tree_name]
                tstptracks = tree["tstpTracks_tracking.centroid_"].array(library="ak")
                tstp_tracks.append(ak.to_numpy(ak.num(tstptracks)))
                luttracks = tree["lutTracks_tracking.centroid_"].array(library="ak")
                lut_tracks.append(ak.to_numpy(ak.num(luttracks)))

        if tstp_tracks:
            tstp_tracks = np.concatenate(tstp_tracks)
        else:
            tstp_tracks = np.array([])

        if lut_tracks:
            lut_tracks = np.concatenate(lut_tracks)
        else:
            lut_tracks = np.array([])
                    
        total_events = len(tstp_tracks)
        clusters_length.append(len(clusters_file)) 
        lut_tracks_made.append(np.count_nonzero(lut_tracks > 0))
        tstp_tracks_made = np.count_nonzero(tstp_tracks > 0)
        lut_fakes_2.append(np.count_nonzero(lut_tracks == 2))
        lut_fakes_3.append(np.count_nonzero(lut_tracks == 3))
        tstp_fakes = np.count_nonzero(tstp_tracks > 1)

    return {
        "total_events": total_events,
        "lut_tracks_made": np.array(lut_tracks_made),
        "tstp_tracks_made": np.array(tstp_tracks_made),
        "lut_fakes_2": np.array(lut_fakes_2),
        "lut_fakes_3": 2 * np.array(lut_fakes_3),
        "tstp_fakes": tstp_fakes
    }

truth = process_thresholds(truth_threshold_files)
notruth = process_thresholds(notruth_threshold_files)

print("----------- TRUTH ---------")
print(f"Total number of events = {truth['total_events']}")
print(f"Total number of cluster triplets = {9 * [number_clusters(truth_file)]}") #multiplied by 9 just to make it a list so it's plottable
print(f"Tracks made by LUT-method = {truth['lut_tracks_made']}")                 #for the 9 (LUT threshold) data points 
print(f"Tracks made by maxdelta method = {truth['tstp_tracks_made']}")
print(f"Fakes made by LUT-method (1 per event) = {truth['lut_fakes_2']}")
print(f"Fakes made by LUT-method (2 per event) = {truth['lut_fakes_3']}")
print(f"Fakes made by maxdelta method = {truth['tstp_fakes']}")

print("\n------- NOTRUTH ---------")
print(f"Total number of events = {notruth['total_events']}")
print(f"Total number of cluster triplets = {9 * [number_clusters(notruth_file)]}")
print(f"Tracks made by LUT-method = {notruth['lut_tracks_made']}")
print(f"Tracks made by maxdelta method = {notruth['tstp_tracks_made']}")
print(f"Fakes made by LUT-method (1 per event) = {notruth['lut_fakes_2']}")
print(f"Fakes made by LUT-method (2 per event) = {notruth['lut_fakes_3']}")
print(f"Fakes made by maxdelta method = {notruth['tstp_fakes']}")

#NOTE: This code assumes that all of your files have successfully produced a clusters.txt, a LUT.txt, a .ROOT file with tracking.
#      If something goes wrong and one of your input files (for whatever reason) successfully produces a clusters.txt but malfunctions
#      at the tracking stage, then this might lead to your data showing that you produced (as an example for 1 mil events), 900.000
#      cluster triplets and 600.000 tracks, which is an unusually low number of tracks. In this case, it's not that fewer tracks were 
#      produced, it's that some files didn't produce any tracks because they didn't successfully complete that step. So just be careful
#      that all events which start the process also fully complete it so that the data doesn't (wrongly) produce unusual results :)

In [ ]:
THP = False #again, you can plot for either the truthfiltered or unfiltered 
            #case by toggling THP (TruthHitProducer)

if THP == True:
    totalevents= truth['total_events'] #total number of starting events
    clusters_length = 9 * [number_clusters(truth_file)] #total number of cluster triplets made
    lut_tracks_made = truth['lut_tracks_made'] #number of tracks made by lut-method
    lut_fakes_2 = truth['lut_fakes_2'] #number of events with 2 tracks made (1 real, 1 fake)
    lut_fakes_3 = truth['lut_fakes_3'] #number of events with 3 tracks made (1 real, 2 fakes)
    tstp_fakes = truth['tstp_fakes'] #number of fakes made by maxdelta method
    tstp_eff = truth['tstp_tracks_made'] / number_clusters(truth_file) #tracking efficiency for maxdelta method
if THP == False:
    totalevents= notruth['total_events']
    clusters_length = 9 * [number_clusters(notruth_file)]
    lut_tracks_made = notruth['lut_tracks_made']
    lut_fakes_2 = notruth['lut_fakes_2'] 
    lut_fakes_3 = notruth['lut_fakes_3'] 
    tstp_fakes = notruth['tstp_fakes']
    tstp_eff = notruth['tstp_tracks_made'] / number_clusters(notruth_file)

if THP == True: 
    insert = "truth"
if THP == False: 
    insert = "non-truth"

print(f"Tracking efficiency of maxdelta method for {insert} case: ", tstp_eff) #this should be the same value printed in a list with 
                                                                               #length equal to the number of LUT thresholds tested

In [ ]:
#this block creates the plot(s)

#this is just for text size
plt.rcParams.update({"font.size": 15, "axes.titlesize": 16,
    "axes.labelsize": 16, "xtick.labelsize": 15,
    "ytick.labelsize": 15, "legend.fontsize": 15})

#these are the threshold values tested
thresholds = np.array([0,0.0002,0.0004,0.0005,0.0008,0.0015,0.005,0.06,0.5])

efficiency = lut_tracks_made / clusters_length
lutfakes = (lut_fakes_2 + lut_fakes_3) / clusters_length

idx = np.argsort(thresholds)
thresholds = thresholds[idx]
efficiency = efficiency[idx]
lutfakes = lutfakes[idx]

thresholds = thresholds[::-1]
efficiency = efficiency[::-1]
lutfakes = lutfakes[::-1]


#the figs are divided into 3 'subplots', one is the fake rate plot (ax3)
#while the upper efficiencies plot is divided into ax1 and ax2 due to the
#y-axis break
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True,
    figsize=(10,8), gridspec_kw={'height_ratios': [11, 1, 3]})

plt.subplots_adjust(hspace=0.05)

d = 0.008
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False, linewidth=1)
ax1.plot((-d, +d), (-d, +d), **kwargs)
ax1.plot((1-d, 1+d), (-d, +d), **kwargs)

#ax1 is the upper efficiencies subplot
ax1.bar(thresholds.astype(str), efficiency,color="slateblue",label="$LUT$")

if THP == True:
    ax1.set_ylim(0.975, 1.001)
if THP == False:
    ax1.set_ylim(0.94, 1.001)
ax1.set_ylabel(r"$\epsilon_{Trk}$")
ax1.grid(axis="y", alpha=0.5)
ax1.spines.bottom.set_visible(False)
ax1.tick_params(labeltop=False)
ax1.text(0.02, 0.77, f"$n_{{Cls}}$ = {number_clusters(truth_file)}",
    transform=ax1.transAxes, verticalalignment="top", horizontalalignment="left",
    bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"))

ax1.axhline(tstp_eff, color="red", linestyle="--", linewidth=1.5, label=f"Baseline={tstp_eff.round(4)}")
ax1.legend(loc="upper left")
#ax1r = ax1.twinx()

#--------------------------------------

ax2.bar(thresholds.astype(str), efficiency, color="slateblue")
if THP == True:
    ax2.set_ylim(0.8, 0.85)
if THP == False:
    ax2.set_ylim(0.75, 0.8)
ax2.grid(axis="y", alpha=0.5)
ax2.spines.top.set_visible(False)
ax2.xaxis.tick_bottom()

kwargs.update(transform=ax2.transAxes)
ax2.plot((-d, +d), (1-d, 1+d), **kwargs)
ax2.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

#fake rates -----------------------------
maxfake = 2.693e-5 #max permitted fake rate 
ax3.axhline(maxfake, color = "lawngreen", linestyle="--", linewidth=2, label="$r_{fake}^{max}$")

ax3.plot(thresholds.astype(str), lutfakes, color="slateblue", marker="s",
    markersize=8, linewidth=2, label="$LUT$") #lut method
ax3.plot(thresholds.astype(str), [tstp_fakes / clusters_length[0]] * len(thresholds), color="orangered",
    marker="o", linewidth=2, linestyle="-", label="$Baseline$") #maxdelta method

if THP == True:
    ax3.set_ylim(-0.01, 0.01)
if THP == False:
    ax3.set_yscale("log")
    ax3.set_ylim(1e-6,1e-2)

ax3.set_ylabel("$r_{fake}$", fontsize=16)
ax3.grid(axis="y", alpha=0.5)
ax3.legend(loc="upper left")

plt.xticks(rotation=30)
plt.xlabel("LUT Threshold")
plt.tight_layout()
plt.show()